# Классификация текстов с помощью предобученного BERT

Задача — бинарная классификация тональности на датасете **SST-2**.

Подход: замороженный `bert-base-uncased` используется как экстрактор признаков,
поверх полученных эмбеддингов обучается логистическая регрессия на PyTorch.

Целевая метрика: не менее **84.5% accuracy** на тестовой части выборки.

## 1. Подготовка окружения

In [1]:
%pip install -q transformers

In [2]:
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, roc_auc_score
from transformers import AutoModel, AutoTokenizer


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

device: cuda


## 2. Загрузка данных

Первые 5000 примеров идут в обучающую выборку, остальные — в тестовую.
Holdout-часть скачивается отдельным файлом.

In [3]:
!wget -q -O texts_holdout.json https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json

In [4]:
data = pd.read_csv(
    "https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv",
    delimiter="\t",
    header=None,
    names=["text", "label"],
)

texts_train = data["text"].values[:5000]
y_train_raw = data["label"].values[:5000]
texts_test = data["text"].values[5000:]
y_test_raw = data["label"].values[5000:]

with open("texts_holdout.json") as iofile:
    texts_holdout = json.load(iofile)

print(f"train:   {len(texts_train)}")
print(f"test:    {len(texts_test)}")
print(f"holdout: {len(texts_holdout)}")
print(f"баланс классов в train: {y_train_raw.mean():.3f}")

train:   5000
test:    1920
holdout: 500
баланс классов в train: 0.521


## 3. Извлечение эмбеддингов

Модель используется только в режиме инференса. В качестве представления текста берётся `pooler_output`.

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased").to(DEVICE)
bert.eval()

n_params = sum(p.numel() for p in bert.parameters())
print(f"параметров в BERT: {n_params / 1e6:.1f}M")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


параметров в BERT: 109.5M


In [6]:
@torch.no_grad()
def get_embeddings(texts, batch_size=32):
    texts = list(texts)
    outputs = []

    for start in range(0, len(texts), batch_size):
        batch = tokenizer(
            texts[start : start + batch_size],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(DEVICE)

        outputs.append(bert(**batch).pooler_output.cpu())

    return torch.cat(outputs, dim=0)

In [7]:
datasets = {
    "train": texts_train,
    "test": texts_test,
    "holdout": texts_holdout,
}

embeddings = {}
for name, texts in datasets.items():
    start_time = time.perf_counter()
    embeddings[name] = get_embeddings(texts)
    elapsed = time.perf_counter() - start_time
    print(f"{name:>7}: {tuple(embeddings[name].shape)} за {elapsed:.1f} с")

torch.save(embeddings, "embeddings.pt")

  train: (5000, 768) за 14.0 с
   test: (1920, 768) за 5.3 с
holdout: (500, 768) за 1.6 с


## 4. Классификатор

Логистическая регрессия поверх замороженных эмбеддингов.
Используется `BCEWithLogitsLoss`, поэтому модель возвращает логиты без сигмоиды.

In [8]:
X_train = embeddings["train"]
X_test = embeddings["test"]
X_holdout = embeddings["holdout"]

y_train = torch.from_numpy(y_train_raw).float().unsqueeze(1)
y_test = torch.from_numpy(y_test_raw).float().unsqueeze(1)

print(f"X_train: {tuple(X_train.shape)}, y_train: {tuple(y_train.shape)}")
print(f"X_test:  {tuple(X_test.shape)}, y_test:  {tuple(y_test.shape)}")

X_train: (5000, 768), y_train: (5000, 1)
X_test:  (1920, 768), y_test:  (1920, 1)


In [9]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim=768):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

In [10]:
def train_model(
    model, features, targets, n_epochs=5000, learning_rate=1e-2, log_every=250
):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = []

    model.train()
    for epoch in range(1, n_epochs + 1):
        optimizer.zero_grad()
        loss = criterion(model(features), targets)
        loss.backward()
        optimizer.step()

        history.append(loss.item())
        if epoch % log_every == 0 or epoch == 1:
            print(f"epoch {epoch:>5} | loss {loss.item():.4f}")

    return history

In [11]:
set_seed()

classifier = LogisticRegression(input_dim=768)
loss_history = train_model(classifier, X_train, y_train)

epoch     1 | loss 0.7945
epoch   250 | loss 0.4367
epoch   500 | loss 0.3870
epoch   750 | loss 0.3642
epoch  1000 | loss 0.3504
epoch  1250 | loss 0.3404
epoch  1500 | loss 0.3325
epoch  1750 | loss 0.3259
epoch  2000 | loss 0.3201
epoch  2250 | loss 0.3149
epoch  2500 | loss 0.3104
epoch  2750 | loss 0.3240
epoch  3000 | loss 0.3030
epoch  3250 | loss 0.3002
epoch  3500 | loss 0.2976
epoch  3750 | loss 0.2953
epoch  4000 | loss 0.2934
epoch  4250 | loss 0.2991
epoch  4500 | loss 0.2897
epoch  4750 | loss 0.2881
epoch  5000 | loss 0.2867


## 5. Оценка качества

In [12]:
@torch.no_grad()
def predict_proba(model, features):
    model.eval()
    return torch.sigmoid(model(features)).squeeze(1)


def evaluate(model, features, targets):
    probabilities = predict_proba(model, features).numpy()
    labels = targets.squeeze(1).numpy()
    predictions = (probabilities >= 0.5).astype(int)

    print(f"accuracy: {accuracy_score(labels, predictions):.4f}")
    print(f"roc-auc:  {roc_auc_score(labels, probabilities):.4f}")

In [13]:
print("train")
evaluate(classifier, X_train, y_train)

print("\ntest")
evaluate(classifier, X_test, y_test)

train
accuracy: 0.8822
roc-auc:  0.9505

test
accuracy: 0.8505
roc-auc:  0.9246


In [14]:
test_accuracy = accuracy_score(
    y_test.squeeze(1).numpy(),
    (predict_proba(classifier, X_test).numpy() >= 0.5).astype(int),
)

status = "порог пройден" if test_accuracy >= 0.845 else "порог не пройден"
print(f"test accuracy = {test_accuracy:.4f} (порог 0.845) — {status}")

test accuracy = 0.8505 (порог 0.845) — порог пройден


In [15]:
probabilities = {
    name: predict_proba(classifier, features).tolist()
    for name, features in (
        ("train", X_train),
        ("test", X_test),
        ("holdout", X_holdout),
    )
}

print({name: len(values) for name, values in probabilities.items()})

{'train': 5000, 'test': 1920, 'holdout': 500}


## 6. Выводы

- Замороженный `bert-base-uncased` + логистическая регрессия дают около **85% accuracy**
  на тестовой выборке, чего достаточно для прохождения порога.
- Лосс на трейне продолжает медленно падать и после 1500 эпох, но качество на тесте
  при этом выходит на плато — дальнейшее обучение линейного слоя даёт переобучение.
- Что можно попробовать дальше: усреднение последнего скрытого слоя вместо `pooler_output`,
  нормализацию признаков, `weight_decay`, а также полный файнтюнинг BERT через
  `AutoModelForSequenceClassification` (обычно 90%+ на SST-2).